In [1]:
from utils.application import *#
from utils.dynamicRieszFunctions import estimateDynamicRiesz_all
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.estimateDiDLinear import estimateDiDLinear
import torch
import pandas as pd
import time
from torch.distributions import Normal
from utils.dgp import DiD_DGP
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

## Settings

lasso_cv_settings = {
    'b_degree' : 1,
    'cv_folds' : 5,
    'random_state' : 42
}

lasso_a_settings = {
    'lambda_val' :0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' : "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

lasso_f_settings = {
    'lambda_val' : 0,
    'beta_start' : None,
    'D_LB' : 0,
    'D_add' : 0.2,
    'c1' :  "CV",
    'c2' : 0.1,
    'tol' : 1e-5,
    'max_iter' : 100,
    'b_degree' : 1,
    'control' : {'maxIter': 1000, 'optTol': 1e-5, 'zeroThreshold': 1e-6}
}

rf_a_settings = {
    'poly_degree' : 0,
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' :1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}
rf_f_settings = {
    'poly_degree' : 1, # 1 or 2?
    'l2' : 0,
    'n_estimators' : 100,
    'criterion' : "mse",
    'max_depth' : None,
    'min_samples_split' : 10,
    'min_samples_leaf' : 5,
    'min_weight_fraction_leaf' : 0.,
    'min_var_fraction_leaf' : None,
    'min_var_leaf_on_val' : False,
    'max_features' : "auto",
    'min_impurity_decrease' : 0.,
    'max_samples' : .45,
    'min_balancedness_tol' : .45,
    'honest' : True,
    'inference' : True,
    'fit_intercept' : True,
    'subforest_size' : 4,
    'n_jobs' :1,
    'random_state' : None,
    'verbose' : 0,
    'warm_start' : False
}

net_a_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}

net_f_settings = {
    'test_split' : 0,
    'learner_lr' : 1e-4,
    'learner_l2' : 1e-3,
    'learner_l1' : 0,
    'n_epochs' : 100,
    'earlystop_rounds' : 20,
    'earlystop_delta' : 1e-3,
    'bs' : 64,
    'optimizer' : 'adam',
    'warm_start' : False,
    'logger' : None,
    'model_dir' : '.',
    'device' : torch.cuda.current_device() if torch.cuda.is_available() else None,
    'n_hidden' : 100,
    'drop_prob' : 0,
    'degree' : 2,
    'interaction_only' : True,
    'n_common' : 200,
    'act_func' : 'elu'
}





In [2]:
seed = 123
folds = 5

data_application = application_data()

In [9]:
data_dict = data_application.get_data(2003, 2004, baseline_2001=False)
X1 = data_dict['X1']
X2 = data_dict['X2']
Y1 = data_dict['Y1']
Y2 = data_dict['Y2']
Z = data_dict['Z']
D = data_dict['D']

Changing covariates:  ['lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4']


In [10]:
Y1.shape

torch.Size([2491, 1])

In [11]:
print("Data shapes:")
print(f"Y1: {Y1.shape}")
print(f"Y2: {Y2.shape}")
print(f"D: {D.shape}")
print(f"Z: {Z.shape}")
print(f"X1: {X1.shape}")
print(f"X2: {X2.shape}")
print()
print("First few values:")
print(f"Y1[0:3]: {Y1[0:3].flatten()}")
print(f"Y2[0:3]: {Y2[0:3].flatten()}")
print(f"D[0:3]: {D[0:3].flatten()}")
print(f"Z[0:3]: {Z[0:3].flatten()}")
print(f"X1[0:3, :5]: {X1[0:3, :5]}")
print(f"X2[0:3, :5]: {X2[0:3, :5]}")

Data shapes:
Y1: torch.Size([2491, 1])
Y2: torch.Size([2491, 1])
D: torch.Size([2491, 1])
Z: torch.Size([2491, 3])
X1: torch.Size([2491, 2])
X2: torch.Size([2491, 2])

First few values:
Y1[0:3]: tensor([5.3891, 5.2781, 8.4615])
Y2[0:3]: tensor([5.3566, 5.2883, 8.3369])
D[0:3]: tensor([0., 0., 0.])
Z[0:3]: tensor([0., 0., 1., 0., 0., 1., 0., 0., 1.])
X1[0:3, :5]: tensor([[ 9.6209, 10.1076],
        [ 9.3381, 10.0718],
        [12.8515, 10.4868]])
X2[0:3, :5]: tensor([[ 9.6265, 10.1403],
        [ 9.3596, 10.1072],
        [12.8689, 10.5367]])


In [12]:
ATT, STD, *_ = estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                     method_a = "RF", rf_a_settings=rf_a_settings,
                                                                        method_f = "RF", rf_f_settings = rf_f_settings, lasso_a_settings= lasso_a_settings, lasso_f_settings=lasso_f_settings)
print("ATT: " ,ATT, f"({STD})")

ATT:  tensor(-0.0152) (0.7705239653587341)


In [13]:
X1[0]

tensor([ 9.6209, 10.1076])

In [14]:
STD/np.sqrt(len(Y1))

tensor(0.0154)

In [ ]:
ATTs, STDs = [], []
data_dict = data_application.get_data(2003, 2004, baseline_2001=False)

for i in tqdm(range(20)):
   X1 = data_dict['X1']
   X2 = data_dict['X2']
   Y1 = data_dict['Y1']
   Y2 = data_dict['Y2']
   Z = data_dict['Z']
   D = data_dict['D']
   ATT, STD, *_ = estimateDynamicRiesz(Y1, Y2, D, Z, X1, X2, folds,
                                                                        method_a = "RF", rf_a_settings=rf_a_settings,
                                                                           method_f = "RF", rf_f_settings = rf_f_settings)
   ATTs.append(ATT)
   STDs.append(STD)


Changing covariates:  ['lpop', 'lavg_pay']
Z variables:  ['region_2', 'region_3', 'region_4']


100%|██████████| 20/20 [04:36<00:00, 13.81s/it]


In [ ]:
# Let's test with the actual settings that are used in the notebook
print("Testing with actual lasso settings:")
print(f"b_degree in lasso_a_settings: {lasso_a_settings['b_degree']}")
print(f"b_degree in lasso_f_settings: {lasso_f_settings['b_degree']}")

# Test with degree 1 (what's actually used)
b_func_1_pred1 = b_polynomial(degree=1)
b_func_1_pred1.set_standardization(predictors_1, D_test)
output_1_deg1 = b_func_1_pred1.b_poly(predictors_1, D_test)

b_func_1_pred2 = b_polynomial(degree=1)  
b_func_1_pred2.set_standardization(predictors_2, D_test)
output_2_deg1 = b_func_1_pred2.b_poly(predictors_2, D_test)

print(f"Degree 1 - predictors_1 output: {output_1_deg1.shape}")
print(f"Degree 1 - predictors_2 output: {output_2_deg1.shape}")

# Let's see what happens with higher degrees
for degree in [2, 3]:
    b_func_deg_pred1 = b_polynomial(degree=degree)
    b_func_deg_pred1.set_standardization(predictors_1, D_test)
    output_deg_pred1 = b_func_deg_pred1.b_poly(predictors_1, D_test)
    
    b_func_deg_pred2 = b_polynomial(degree=degree)
    b_func_deg_pred2.set_standardization(predictors_2, D_test)
    output_deg_pred2 = b_func_deg_pred2.b_poly(predictors_2, D_test)
    
    print(f"Degree {degree} - predictors_1 output: {output_deg_pred1.shape}")
    print(f"Degree {degree} - predictors_2 output: {output_deg_pred2.shape}")
    print(f"Degree {degree} - difference: {output_deg_pred2.shape[1] - output_deg_pred1.shape[1]}")

    # Check if 13 and 9 appear anywhere
    if output_deg_pred1.shape[1] == 13 or output_deg_pred1.shape[1] == 9:
        print(f"Found dimension 13 or 9 in degree {degree} predictors_1!")
    if output_deg_pred2.shape[1] == 13 or output_deg_pred2.shape[1] == 9:
        print(f"Found dimension 13 or 9 in degree {degree} predictors_2!")